# 00 — Business Understanding

**Project:** Health Insurance Cross Sell  
**Goal:** Rank customers by propensity to buy vehicle insurance so the sales team maximises conversions within a fixed call capacity.

---

## 1. Context

An insurance company currently provides **health insurance** to hundreds of thousands of customers. The company wants to expand revenue by cross-selling **vehicle insurance** to the same customer base.

The sales team has a **limited number of phone calls** they can make each campaign period. Calling customers at random would waste most of those calls on people with no interest in vehicle insurance (only ~12 % of the base is interested).

**The business question is:**
> Given a list of customers, in which order should the sales team call them to maximise the number of conversions?

This is a **ranking / propensity-scoring** problem, not a pure classification problem. The model output is a continuous score that determines call priority.

## 2. Business Assumptions

| # | Assumption |
|---|---|
| 1 | The sales team can call at most **20 000 customers** per campaign period. |
| 2 | A customer who responds `1` in the survey is considered **genuinely interested** and will convert if contacted. |
| 3 | Contacting an uninterested customer has negligible marginal cost. |
| 4 | Every converted customer generates a fixed revenue of approximately USD 1 500. |
| 5 | The historical survey data (train.csv) is representative of future customers. |

## 3. Solution Strategy

The project follows a **CRISP-DM** cycle:

1. **Business Understanding** ← *you are here*
2. **Data Understanding** — explore the raw dataset
3. **Exploratory Data Analysis** — test hypotheses with visualisations
4. **Feature Engineering** — encode and transform raw columns
5. **Preprocessing Pipeline** — `ColumnTransformer` (imputation + scaling + OHE)
6. **Model Comparison** — cross-validate Logistic Regression, Random Forest, LightGBM
7. **Hyperparameter Tuning** — `RandomizedSearchCV` on the best model
8. **Business Evaluation** — cumulative gain curve, lift curve, revenue estimate
9. **Deployment** — FastAPI endpoint + Google Sheets automation
10. **Monitoring** — track score distribution drift in production

## 4. Success Metrics

### 4.1 ROC AUC

$$\text{ROC AUC} = P(\text{score}_{positive} > \text{score}_{negative})$$

- Measures the probability that a randomly chosen positive customer is scored higher than a randomly chosen negative customer.
- **Threshold-agnostic** — useful because we are ranking, not classifying.
- Target: **> 0.85**.

### 4.2 Average Precision (AP)

$$\text{AP} = \sum_k (R_k - R_{k-1}) \cdot P_k$$

- Area under the Precision-Recall curve.
- More sensitive to performance at the **top of the ranked list** than ROC AUC.
- Particularly important with imbalanced data (~12 % positive rate).

### 4.3 Precision @ 20 000

$$\text{Precision@K} = \frac{\text{true positives in top } K}{K}$$

- Of the 20 000 customers we call, what fraction is actually interested?
- Direct measure of call efficiency.

### 4.4 Lift @ 20 000

$$\text{Lift@K} = \frac{\text{Precision@K}}{\text{baseline positive rate}}$$

**Intuitive explanation:**  
If 12 % of customers are interested (baseline), and our model's top 20 000 have a 37 % positive rate, then:

$$\text{Lift} = \frac{0.37}{0.12} \approx 3.1$$

This means the model is **3.1× more efficient** than random calling. If the company makes 20 000 calls:
- Random: reaches ~2 400 interested customers
- Model: reaches ~7 400 interested customers

Target: **Lift@20 000 > 2.5**.

## 5. Deliverables

| Deliverable | Description |
|---|---|
| `models/model.joblib` | Trained sklearn Pipeline (preprocess + model) |
| `reports/metrics.json` | Final evaluation metrics |
| `data/processed/predictions.csv` | Scored test set, sorted by score descending |
| FastAPI endpoint | `/predict` endpoint for real-time or batch scoring |
| Google Sheets script | One-click scoring from a spreadsheet |